# DynaFX — Run & Visualize a Simulation in Jupyter

This notebook demonstrates the core DynaFX workflow entirely in Python:

1. Build a system-dynamics model with the **Python-native DSL** (or parse a `.sysd` file).
2. **Simulate** it.
3. **Visualize** the results with **Plotly** (interactive time series) and
   **NetworkX** (model structure / stock-flow diagram).

Everything runs against the **`dynafx` wheel** — no source checkout required.

```python
pip install "https://github.com/Achref-Yak/DynaFX/releases/download/v0.2.0/dynafx-0.2.0-py3-none-any.whl"
```


## 0. Imports

DynaFX ships with `plotly` and `networkx` as dependencies, so they are already
available after installing the wheel.


## 1. Build a model with the Python DSL

Define stocks, their flows, and auxiliary variables programmatically.
Here is a classic two-stock **predator–prey** (Lotka–Volterra) model.


## 2. Simulate

`SysdModel.simulate()` returns a `SysdModelResult` with the full trajectory.


## 3. Interactive time series with Plotly

`SysdModelResult` exposes `times`, `values` (per stock) and `aux_values`.


## 4. Raw data (pandas)

Simulation specialists usually want the *data*, not just a picture.
Assemble a `DataFrame` for further analysis.


## 5. Model structure with NetworkX

`SysdModel.to_decomposer()` produces a `Graph` of stocks, flows and the causal
edges between them. We can inspect it directly, or lay it out with Plotly.


A small helper renders the stock–flow graph as an interactive Plotly network,
so structure and dynamics live side by side in one notebook.


## 6. Same flow, from a `.sysd` file

You don't have to build models in Python. `parse_sysd` reads the same
structure from a `.sysd` source string (and `parse_sysd_file` reads a file).


In [1]:
import dynafx
import plotly
import networkx
import networkx as nx
import plotly.graph_objects as go
import pandas as pd

from dynafx import SysdModel, parse_sysd

print("dynafx  :", dynafx.__file__)
print("plotly  :", plotly.__version__)
print("networkx:", networkx.__version__)


dynafx  : /tmp/opencode/nb-venv/lib/python3.12/site-packages/dynafx/__init__.py
plotly  : 6.9.0
networkx: 3.6.1


In [2]:
model = SysdModel("predator_prey")
model.dt = 0.05
model.t_span = (0.0, 50.0)

with model.stock("prey", 20.0) as s:
    s.inflow("births", "0.8 * prey")
    s.outflow("predation", "0.08 * prey * predator")

with model.stock("predator", 8.0) as s:
    s.inflow("reproduction", "0.02 * prey * predator")
    s.outflow("deaths", "0.4 * predator")

# an auxiliary: per-prey predation rate
model.aux("predation_rate", "0.08 * prey")

print("stocks:", [s.name for s in model.stocks])
print("auxes :", [a.name for a in model.aux_vars])


stocks: ['prey', 'predator']
auxes : ['predation_rate']


In [3]:
result = model.simulate()

print("method:", result.method, "| steps:", result.steps)
print("t span:", result.times[0], "->", result.times[-1])
print("final prey    :", round(result.values["prey"][-1], 2))
print("final predator:", round(result.values["predator"][-1], 2))


method: rk4 | steps: 1000
t span: 0.0 -> 49.9999999999993
final prey    : 19.68
final predator: 12.3


In [4]:
fig = go.Figure()
for name in result.stocks:
    fig.add_trace(go.Scatter(
        x=result.times, y=result.values[name], mode="lines", name=name,
    ))
fig.add_trace(go.Scatter(
    x=result.times, y=result.aux_values["predation_rate"],
    mode="lines", name="predation_rate", line=dict(dash="dot"),
))
fig.update_layout(
    title="Predator–Prey over time",
    xaxis_title="Time",
    yaxis_title="Population",
    hovermode="x unified",
)
fig.show()


In [5]:
df = pd.DataFrame({"time": result.times})
for name in result.stocks:
    df[name] = result.values[name]
for name in result.aux_values:
    df["aux_" + name] = result.aux_values[name]

print(df.shape)
df.head()


(1001, 4)


,time,prey,predator,aux_predation_rate
0,0.00,20.000000,8.000000,1.600000
1,0.05,20.160624,8.000642,1.612850
2,0.10,20.322434,8.002574,1.625795
3,0.15,20.485332,8.005806,1.638827
4,0.20,20.649213,8.010348,1.651937


In [6]:
g = model.to_decomposer().graph   # DynaFX Graph

# Convert to a NetworkX DiGraph keyed by node id
G = nx.DiGraph()
for n in g.nodes.values():
    G.add_node(str(n.id), label=n.text, kind=n.type.name)
for e in g.edges.values():
    G.add_edge(str(e.source_id), str(e.target_id), polarity=e.polarity)

print("nodes:", G.number_of_nodes(), "| edges:", G.number_of_edges())
for nid, data in G.nodes(data=True):
    print(f"  {data['kind']:6s} {data['label']}")
print("edges (source -> target, polarity):")
for u, v, data in G.edges(data=True):
    print(f"  {G.nodes[u]['label']} -> {G.nodes[v]['label']} ({data['polarity']:+d})")


nodes: 6 | edges: 4
  STOCK  prey
  STOCK  predator
  FLOW   births
  FLOW   predation
  FLOW   reproduction
  FLOW   deaths
edges (source -> target, polarity):
  births -> prey (+1)
  predation -> prey (-1)
  reproduction -> predator (+1)
  deaths -> predator (-1)


In [7]:
pos = nx.spring_layout(G, seed=7, k=1.6)

node_x = [pos[nid][0] for nid in G.nodes]
node_y = [pos[nid][1] for nid in G.nodes]
node_text = [G.nodes[nid]["label"] for nid in G.nodes]

edge_x, edge_y = [], []
for u, v in G.edges:
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

net = go.Figure()
net.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(color="#9ca3af", width=1.5), hoverinfo="none",
))
net.add_trace(go.Scatter(
    x=node_x, y=node_y, mode="markers+text",
    text=node_text, textposition="top center",
    marker=dict(
        size=28,
        color=["#6366f1" if G.nodes[nid]["kind"] == "STOCK" else "#14b8a6"
               for nid in G.nodes],
    ),
    hovertemplate="%{text}<extra></extra>",
))
net.update_layout(
    title="Stock–Flow structure",
    showlegend=False,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
    width=700, height=450,
)
net.show()


In [8]:
sysd_source = """\
// Simple population with carrying capacity
dt 0.5
from 0 to 200

aux growth_rate: 0.10
aux capacity: 1000.0

stock population: 10.0
  + births: growth_rate * population * (1 - population / capacity)
"""

m2 = parse_sysd(sysd_source)
r2 = m2.simulate()

fig2 = go.Figure(go.Scatter(
    x=r2.times, y=r2.values["population"], mode="lines", name="population",
))
fig2.update_layout(
    title="Logistic growth (parsed from .sysd text)",
    xaxis_title="Time", yaxis_title="Population",
    hovermode="x unified",
)
fig2.show()

print("final population:", round(r2.values["population"][-1], 2))


final population: 1000.0
